In [0]:
# -------------------------------------------------------------------------
# PHASE 2: SERVING & BI LAYER (STAR SCHEMA MODELING)
# -------------------------------------------------------------------------
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime
from pyspark.sql import SparkSession

# Initialize our local Spark context for relational views
spark = SparkSession.builder.getOrCreate()

# Dynamically find the latest processed file from Phase 1
PROCESSED_PATH = "./worldbank_data/processed"
search_pattern = f"{PROCESSED_PATH}/sovereign_panel_clean_*.csv"
list_of_files = glob.glob(search_pattern)

if not list_of_files:
    raise FileNotFoundError(f"No processed files found matching pattern: {search_pattern}")

# Sort files to find the newest one
latest_csv_path = max(list_of_files, key=os.path.getctime)

# Load the data panel into memory
df_panel = pd.read_csv(latest_csv_path)
print(f"✓ Successfully ingested {len(df_panel)} cleaned historical records from Phase 1.")
print(f"✓ Source file: {latest_csv_path}")

In [0]:
# -------------------------------------------------------------------------
# STEP 2: DIMENSIONAL MODELING (STAR SCHEMA SETUP)
# -------------------------------------------------------------------------

# 1. Create the Dimension Table (dim_country)
dim_country = df_panel[["country_code", "country_name"]].drop_duplicates().reset_index(drop=True)
dim_country["region_group"] = np.where(
    dim_country["country_code"].isin(["US", "MX", "BR", "AR"]), "Americas", 
    np.where(dim_country["country_code"].isin(["GB", "DE", "GR", "TR"]), "Europe/ME", "Asia/Pacific")
)

# 2. Create the Fact Table (fact_sovereign_metrics)
# Fixed the column name from debt_service_exports to debt_service_pct
fact_cols = [
    "country_code", "year", "gdp_usd", "gdp_growth_pct", 
    "debt_to_gdp_pct", "inflation_pct", "unemployment_pct", 
    "current_account_gdp", "gdp_per_capita_usd",
    "is_complete_record"
]
fact_sovereign_metrics = df_panel[fact_cols].copy()

print(f"✓ Dimension Table 'dim_country' created: {dim_country.shape[0]} unique entities.")
print(f"✓ Fact Table 'fact_sovereign_metrics' created: {fact_sovereign_metrics.shape[0]} historical metrics entries.")

In [0]:
# -------------------------------------------------------------------------
# STEP 3: DERIVED BUSINESS METRICS
# -------------------------------------------------------------------------

# 1. Calculate Misery Index (economic hardship indicator)
# Misery Index = Inflation Rate + Unemployment Rate
fact_sovereign_metrics["misery_index"] = (
    fact_sovereign_metrics["inflation_pct"].fillna(0) + 
    fact_sovereign_metrics["unemployment_pct"].fillna(0)
)

# 2. Calculate Debt Sustainability Signal (risk classification)
# Based on IMF/World Bank thresholds for debt-to-GDP ratios
def classify_debt_risk(debt_to_gdp):
    if pd.isna(debt_to_gdp):
        return "Unknown"
    elif debt_to_gdp < 60:
        return "Sustainable"
    elif debt_to_gdp < 90:
        return "Moderate Risk"
    else:
        return "High Risk"

fact_sovereign_metrics["debt_sustainability_signal"] = (
    fact_sovereign_metrics["debt_to_gdp_pct"].apply(classify_debt_risk)
)

# 3. Display summary statistics
print(f"✓ Derived metrics added to fact table.")
print(f"  - Misery Index range: {fact_sovereign_metrics['misery_index'].min():.2f} to {fact_sovereign_metrics['misery_index'].max():.2f}")
print(f"  - Debt Risk Distribution:")
print(fact_sovereign_metrics["debt_sustainability_signal"].value_counts().to_string())

In [0]:
# -------------------------------------------------------------------------
# STEP 4: REGISTER RELATIONAL VIEW LAYER
# -------------------------------------------------------------------------

# Convert our Pandas Star Schema DataFrames into Spark DataFrames
spark_dim_country = spark.createDataFrame(dim_country)
spark_fact_sovereign = spark.createDataFrame(fact_sovereign_metrics)

# Register them as temporary SQL views for query consumption
spark_dim_country.createOrReplaceTempView("v_dim_country")
spark_fact_sovereign.createOrReplaceTempView("v_fact_sovereign_metrics")

print("✓ Relational View Layer Registered Successfully!")
print("  - SQL View available: v_dim_country")
print("  - SQL View available: v_fact_sovereign_metrics")

In [0]:
# -------------------------------------------------------------------------
# STEP 4.5: SAVE TO PERMANENT UNITY CATALOG TABLES
# -------------------------------------------------------------------------

# Save dimension table to Unity Catalog
spark_dim_country.write.mode("overwrite").saveAsTable("workspace.default.dim_country")
print("✓ Permanent table created: workspace.default.dim_country")

# Save fact table to Unity Catalog
spark_fact_sovereign.write.mode("overwrite").saveAsTable("workspace.default.fact_sovereign_metrics")
print("✓ Permanent table created: workspace.default.fact_sovereign_metrics")

# Verify the tables were created
print("\n📊 Table Metadata:")
print(f"  - dim_country: {spark.table('workspace.default.dim_country').count()} rows")
print(f"  - fact_sovereign_metrics: {spark.table('workspace.default.fact_sovereign_metrics').count()} rows")
print("\n✓ Tables are now accessible from SQL warehouses and dashboards!")

In [0]:
# -------------------------------------------------------------------------
# STEP 5: INTERACTIVE NOTEBOOK WIDGETS
# -------------------------------------------------------------------------

# 1. Fetch unique country codes and names for our dropdown options
country_options = df_panel["country_code"].unique().tolist()
country_options.sort()

# 2. Clear out any existing widgets to prevent layout overlap
dbutils.widgets.removeAll()

# 3. Create the dropdown widgets at the top of the notebook
dbutils.widgets.dropdown("Select_Country", "US", country_options, "1. Target Country")
dbutils.widgets.dropdown("Select_Region", "All", ["All", "Americas", "Europe/ME", "Asia/Pacific"], "2. Macro Region")

# 4. Capture the current interactive selections into local python variables
selected_country = dbutils.widgets.get("Select_Country")
selected_region = dbutils.widgets.get("Select_Region")

print(f"✓ Interactive widgets initialized!")
print(f"  - Currently filtering by Country: {selected_country}")
print(f"  - Currently filtering by Region: {selected_region}")

In [0]:
# -------------------------------------------------------------------------
# STEP 6: DYNAMIC QUERY LAYER (CONNECTING WIDGETS TO DATA)
# -------------------------------------------------------------------------

# Read the active selections from the widgets at the top of the screen
selected_country = dbutils.widgets.get("Select_Country")
selected_region = dbutils.widgets.get("Select_Region")

# Build a dynamic SQL query based on widget criteria
if selected_region == "All":
    # Filter strictly by the chosen country dropdown
    query = f"""
        SELECT f.year, d.country_name, d.region_group, f.gdp_growth_pct, 
               f.debt_to_gdp_pct, f.misery_index, f.debt_sustainability_signal
        FROM v_fact_sovereign_metrics f
        JOIN v_dim_country d ON f.country_code = d.country_code
        WHERE f.country_code = '{selected_country}'
        ORDER BY f.year DESC
    """
else:
    # Scale out to show the entire chosen macro region
    query = f"""
        SELECT f.year, d.country_name, d.region_group, f.gdp_growth_pct, 
               f.debt_to_gdp_pct, f.misery_index, f.debt_sustainability_signal
        FROM v_fact_sovereign_metrics f
        JOIN v_dim_country d ON f.country_code = d.country_code
        WHERE d.region_group = '{selected_region}'
        ORDER BY d.country_name ASC, f.year DESC
    """

# Execute the query in Spark SQL and display an interactive data grid
df_filtered = spark.sql(query)
display(df_filtered)

In [0]:
processed_csv_path = f"{PROCESSED_PATH}/sovereign_panel_clean_{today}.csv"

# 1. Save the cleaned data into the processed folder
cleaned_panel.to_csv(processed_csv_path, index=False)
print(f"✓ Cleaned dataset saved to: {processed_csv_path}")

# 2. Add lineage metadata and convert to Spark DataFrame
cleaned_panel["source"] = "WORLD_BANK"
spark_df = spark.createDataFrame(cleaned_panel)

# 3. Save to Delta Table safely with local session fallback
try:
    spark_df.write.mode("overwrite").saveAsTable("bronze.worldbank_sovereign_raw")
    print("✓ Bronze Delta table created successfully!")
except Exception as e:
    print("⚠ DBFS Root restricted. Registering local temporary Spark view instead...")
    spark_df.createOrReplaceTempView("worldbank_sovereign_raw")
    print("✓ Temporary View created: worldbank_sovereign_raw")

print(f"  Final Table Rows: {spark_df.count()} | Unique Countries: {cleaned_panel['country_code'].nunique()}")

In [0]:
latest = (
    panel.sort_values("year")
         .groupby("country_code")
         .last()
         .reset_index()
)

display(
    latest[[
        "country_name", "year", 
        "gdp_growth_pct", "debt_to_gdp_pct", 
        "inflation_pct", "misery_index", 
        "debt_sustainability_signal"
    ]].sort_values("misery_index", ascending=False)
)